In [ ]:
!pip uninstall -y torch torchvision torchaudio numpy scipy lightning torchmetrics pyannote.audio whisperx onnxruntime torchcodec -q
!pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128 -q
!pip install numpy==2.1.2 -q
!apt-get update -qq
!apt-get install -y ffmpeg -qq
!pip install whisperx==3.8.1 -q
!echo "DONE - RESTART RUNTIME AFTER THIS CELL"

In [ ]:
!pip install fastapi uvicorn
!pip install pyngrok
!pip install nest-asyncio

In [ ]:
import whisperx
import gc
from whisperx.diarize import DiarizationPipeline
import os


!ngrok authtoken 
os.environ["HUGGINGFACE_HUB_TOKEN"] = ""

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [7]:
device = "cuda"
audio_file = "/content/drive/MyDrive/test.wav"
batch_size = 16
compute_type = "float16"

In [ ]:
model = whisperx.load_model("large-v2", device, compute_type=compute_type)
diarize_model = DiarizationPipeline(token="", device=device)
# audio = whisperx.load_audio(audio_file)

2026-02-25 14:18:41 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-02-25 14:18:41 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


2026-02-25 14:18:41 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


In [18]:
import torch
def clean_for_json(obj):
    if isinstance(obj, dict):
        return {k: clean_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.generic):
        return obj.item()
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    else:
        return obj

In [19]:
import io
import copy
import soundfile as sf
import numpy as np
from fastapi import FastAPI, Body
from fastapi.responses import JSONResponse
import uvicorn
import threading
import subprocess
import time
import tempfile

app = FastAPI(title="STT API")

@app.get("/")
async def home():
    return {"stt server": "hello"}

@app.post("/stt")
async def stt_endpoint(audio_wav: bytes = Body(...)):
    audio = None
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp:
        tmp.write(audio_wav)
        tmp.flush()

        audio = whisperx.load_audio(tmp.name)

    result = model.transcribe(audio, batch_size=batch_size)

    rs_1 = copy.deepcopy(result)
    print(rs_1)

    model_a, metadata = whisperx.load_align_model(
        language_code=result["language"],
        device=device
    )

    result = whisperx.align(
        result["segments"],
        model_a,
        metadata,
        audio,
        device,
        return_char_alignments=False
    )
    rs_2 = copy.deepcopy(result)
    print(rs_2)
    diarize_segments = diarize_model(audio)
    result = whisperx.assign_word_speakers(diarize_segments, result)

    rs_3 = copy.deepcopy(result)
    print(rs_3)

    clean_data = clean_for_json([rs_1, rs_2, rs_3])
    return JSONResponse(content=clean_data)

PORT = 1001

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

ngrok_proc = subprocess.Popen(
    ["ngrok", "http", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("FastAPI + ngrok đang chạy")

INFO:     Started server process [26268]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:1001 (Press CTRL+C to quit)


FastAPI + ngrok đang chạy


In [13]:
audio

array([ 0.00027466,  0.00054932, -0.00024414, ..., -0.00286865,
       -0.00427246, -0.00527954], shape=(69120,), dtype=float32)

In [14]:
audio_np

NameError: name 'audio_np' is not defined

In [29]:
rs_1

{'segments': [{'text': ' Alo alo, có gì vui không? Kể tôi nghe với bạn là cái đồ quần què đúng không?',
   'start': 0.031,
   'end': 4.334}],
 'language': 'vi'}

In [30]:
rs_2

{'segments': [{'start': 0.031,
   'end': 0.9,
   'text': ' Alo alo, có gì vui không?',
   'words': [{'word': 'Alo',
     'start': 0.031,
     'end': 0.112,
     'score': np.float64(0.014)},
    {'word': 'alo,', 'start': 0.132, 'end': 0.213, 'score': np.float64(0.015)},
    {'word': 'có', 'start': 0.233, 'end': 0.273, 'score': np.float64(0.014)},
    {'word': 'gì', 'start': 0.334, 'end': 0.395, 'score': np.float64(0.01)},
    {'word': 'vui', 'start': 0.415, 'end': 0.475, 'score': np.float64(0.01)},
    {'word': 'không?',
     'start': 0.496,
     'end': 0.9,
     'score': np.float64(0.012)}]},
  {'start': 0.92,
   'end': 4.354,
   'text': 'Kể tôi nghe với bạn là cái đồ quần què đúng không?',
   'words': [{'word': 'Kể',
     'start': 0.92,
     'end': 0.96,
     'score': np.float64(0.01)},
    {'word': 'tôi', 'start': 0.98, 'end': 1.041, 'score': np.float64(0.013)},
    {'word': 'nghe', 'start': 1.324, 'end': 1.607, 'score': np.float64(0.012)},
    {'word': 'với', 'start': 1.667, 'end': 

In [ ]:
rs_3